# Lab 11: Attention


### Attention is a mechanism that allows a neural network to focus more on relevant information when making a prediction.

Encoder-decoder architecture with attention:
<img src="https://drive.google.com/thumbnail?id=1-QeHdZIeKTo_nAqfh7pRA_o4Nk8CAmP6&sz=w1000" width="700">

## Step 1 — Import libraries

In [ ]:
import pandas as pd
import torch
from torch import nn
import matplotlib.pyplot as plt
from pathlib import Path

torch.manual_seed(42)


## Step 2 — Upload the dataset

In [ ]:
dataset_file = Path("dataset/Lab_11_QnA_Dataset.csv")
if not dataset_file.exists():
    dataset_file = Path("LAB11/dataset/Lab_11_QnA_Dataset.csv")

df = pd.read_csv(dataset_file)
print("Number of questions:", len(df))
print("Number of answer groups:", df["answer_id"].nunique())
display(df.head())


## Step 3 — Preprocessing

1. Lowercase each question and remove punctuation.
2. Give each word a number.
3. Make all questions the same length by adding zeros (**padding**).

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. Lowercase
df["question"] = df["question"].str.lower()

# 2. Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts(df["question"])

# Convert words to numbers and add padding
X = pad_sequences(tokenizer.texts_to_sequences(df["question"]), padding="post")

# Answers
y = df["answer_id"].values

# Convert to PyTorch tensors
X = torch.tensor(X, dtype=torch.long)
y = torch.tensor(y, dtype=torch.long)

## Step 4 — Build an RNN with attention

- **Embedding** turns word numbers into vectors.
- **RNN** reads the words in sequence.
- **Attention** gives each word a weight.
- **Output** chooses one answer from the dataset.

In [ ]:
class RNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(len(tokenizer.word_index) + 1, 16, padding_idx=0)
        self.rnn = nn.RNN(16, 32, batch_first=True)
        self.att = nn.Linear(32, 1)
        self.fc = nn.Linear(32, df["answer_id"].nunique())

    def forward(self, x):
        out, _ = self.rnn(self.emb(x))
        scores = self.att(out).squeeze(-1)
        scores = scores.masked_fill(x == 0, -1e9)
        weights = torch.softmax(scores, dim=1)
        context = (out * weights.unsqueeze(-1)).sum(dim=1)
        return self.fc(context), weights

model = RNN()


## Step 5 — Train the model
The model learns which answer belongs to each question.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(50):
    model.train()
    optimizer.zero_grad()
    predictions, weights = model(X)
    loss = nn.functional.cross_entropy(predictions, y)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1}: Loss = {loss.item():.4f}")

print("Training finished!")


## Step 6 — See attention
The bar chart shows how much attention the model gives to each question word.

In [ ]:
question = df["question"].iloc[0]
words = tokenizer.sequences_to_texts([X[0].tolist()])[0].split()

model.eval()
with torch.no_grad():
    prediction, weights = model(X[:1])

plt.bar(words, weights[0, :len(words)].tolist())
plt.title("Attention Weights")
plt.xticks(rotation=45)
plt.show()


## Tasks:


Task 1 — Understand Attention Weights
Easy

Modify Step 6 to visualize the attention weights for three different questions from the dataset instead of only the first question.

Instructions:
- Select three questions from the dataset.
- Generate an attention bar chart for each question.
- Identify the word receiving the highest attention weight in each question.

Question: Do the words with the highest attention weights appear important for understanding the question? Explain your observations.

In [ ]:
# Task 1 - visualize three questions
question_indices = [0, len(df) // 2, len(df) - 1]
model.eval()

for index in question_indices:
    words = tokenizer.sequences_to_texts([X[index].tolist()])[0].split()
    with torch.no_grad():
        prediction, weights = model(X[index:index+1])
    values = weights[0, :len(words)].tolist()
    top_word = words[values.index(max(values))]

    print("Question:", df["question"].iloc[index])
    print("Highest attention:", top_word)
    plt.figure(figsize=(7, 3))
    plt.bar(words, values)
    plt.title(f"Question {index + 1}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


**Task 1 — Observation:** The tallest bar is the word with the highest attention weight. Some top words may help identify the question topic, while others may be common words. The result depends on what the small model learned from this dataset.


Task 2 — Compare Attention for Similar Questions
Medium

Select two questions from the dataset that have similar meanings but different wording.

Instructions:
- Pass both questions through the trained model.
- Visualize their attention weights.
- Compare the words receiving the highest attention weights.

Question: Does the model focus on similar or different words when processing questions with similar meanings? What does this suggest about how attention works?

In [ ]:
# Task 2 - similar questions about breathing
similar_indices = df.index[df["question"].str.contains("breathe", case=False, na=False)].tolist()[:2]
assert len(similar_indices) == 2, "Could not find two questions about breathing."

top_words = []
model.eval()
for index in similar_indices:
    words = tokenizer.sequences_to_texts([X[index].tolist()])[0].split()
    with torch.no_grad():
        prediction, weights = model(X[index:index+1])
    values = weights[0, :len(words)].tolist()
    top_word = words[values.index(max(values))]
    top_words.append(top_word)

    print("Question:", df["question"].iloc[index])
    print("Answer group:", prediction.argmax(dim=1).item())
    print("Highest attention:", top_word)
    plt.figure(figsize=(7, 3))
    plt.bar(words, values)
    plt.title(df["question"].iloc[index])
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

print("Same highest-attention word:", top_words[0] == top_words[1])


**Task 2 — Observation:** The two questions are both about breathing but use different wording. Their top attention words may be the same or different. Attention assigns weights based on the input words and the learned model, so related questions do not have to produce identical weights.
